# 라이브러리 및 데이터 불러오기

In [1]:
import pandas as pd
import numpy as np
import ast

In [2]:
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"

client = bigquery.Client(project=PROJECT_ID)

In [3]:
# 전처리 수행 대상 테이블 호출
sql = f"""
    SELECT * 
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_blockrecord`
"""

# 판다스 데이터프레임으로 변환
df = client.query(sql).to_dataframe()

df.head()

c:\workspace\final_project\sns_service_analysis\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,id,reason,created_at,block_user_id,user_id
0,1,그냥...,2023-05-04 23:01:53+00:00,867483,878476
1,6,그냥...,2023-05-05 05:21:52+00:00,883696,883511
2,7,그냥...,2023-05-05 06:40:34+00:00,871349,870177
3,14,그냥...,2023-05-05 13:04:52+00:00,885794,879662
4,21,그냥...,2023-05-05 15:36:34+00:00,887434,881108


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19482 entries, 0 to 19481
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype              
---  ------         --------------  -----              
 0   id             19482 non-null  Int64              
 1   reason         19482 non-null  str                
 2   created_at     19482 non-null  datetime64[us, UTC]
 3   block_user_id  19482 non-null  Int64              
 4   user_id        19482 non-null  Int64              
dtypes: Int64(3), datetime64[us, UTC](1), str(1)
memory usage: 1.2 MB


- 다른 테이블도 created_at 날짜가 모두 동일한 UTC기준인지 확인 필요.

## 중복 체크
- 전체 중복 및 유저 아이디 기준 중복체크 결과.
유저단위의 테이블이 아닌 신고 개별 건수가 기록되는 테이블로 확인되어 유저 아이디 기준 중복은 따로 처리하지 않음.

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df['user_id'].duplicated().sum()

np.int64(6930)

## 결측 추가 체크
- info() 결과 null값은 확인되지 않았으며, 추가 체크에서도 결측으로 존재하는 값은 없다고 확인.

In [8]:
df.isna().sum()

id               0
reason           0
created_at       0
block_user_id    0
user_id          0
dtype: int64

## reason 컬럼의 이상값 체크
- 사유가 이상한 데이터가 있는지 확인.
- 각 사유에 대한 비율 확인.

In [9]:
df['reason'].value_counts()

reason
모르는 사람임               9640
친구 사이가 어색해짐           5805
사칭 계정                 2022
나랑 관련 없는 질문을 자꾸 보냄    1083
너무 많은 양의 질문을 보냄        919
기타                       7
그냥...                    6
Name: count, dtype: int64

## 신고한, 신고 당한 유저아이디 체크
- 각각의 유저아이디의 이상값을 확인한 결과.
모두 숫자형 데이터로 최소값과 최대값으로 범위를 확인하여 유저 테이블의 아이디와 비교해본 결과 특이값이 없다고 판단

In [10]:
print(df['user_id'].min())
print(df['user_id'].max())

837615
1583612


In [12]:
df[df['user_id'] == 1583612]

,id,reason,created_at,block_user_id,user_id
11356,25360,모르는 사람임,2024-05-05 11:06:31+00:00,1582869,1583612


In [14]:
df['block_user_id'].nunique()

16240

In [15]:
print(df['block_user_id'].min())
print(df['block_user_id'].max())

832740
1582869


- reason의 값이 이상하거나 수치적으로 문제가 있어보이지 않음.
- 신고 당한 유저의 고유 수 16,240명

In [17]:
df[df['block_user_id'] == 1582869]

,id,reason,created_at,block_user_id,user_id
11356,25360,모르는 사람임,2024-05-05 11:06:31+00:00,1582869,1583612


In [19]:
df['block_user_id'].value_counts().head(10)

block_user_id
898020     76
897681     25
877266     25
1380465    25
876207     24
1395312    24
1198628    21
1495281    19
1031842    18
1186363    17
Name: count, dtype: Int64

## 정리
- 모든 테이블의 날짜 데이터의 기준은 확인필요.
- 다른 전처리 작업은 없음. 그대로 유지하는 것을 결론으로 수정 및 처리 코딩없이 유지함.